# Nhiệm vụ 7: Đánh giá chất lượng cụm (PCA + K-Means)

## 1. Đọc dữ liệu Diagnostics

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Đặt thư mục làm việc về gốc dự án an toàn
project_root = r'C:\Users\HOME\Desktop\M2\Risk-Adjusted-Momentum-Clustering-m2-clustering-experiment'
if os.path.exists(project_root):
    os.chdir(project_root)

print('Current Working Directory:', os.getcwd())

diag_path = 'M2/artifacts/m2-task6-pca-kmeans-v1/diagnostics.csv'
df = pd.read_csv(diag_path)

# Lọc K=2 (K-Means Baseline được chốt ở K=2)
df_k2 = df[df['k'] == 2].copy()
df_k2['snapshot_date'] = pd.to_datetime(df_k2['snapshot_date'])
df_k2.sort_values('snapshot_date', inplace=True)
display(df_k2.head())


## 2. BẢNG 1: Tóm tắt Thống kê (Quality Summary)

In [ ]:
out_dir = 'M2/artifacts/m2-evaluation-pca-kmeans'
os.makedirs(out_dir, exist_ok=True)

metrics_order = ["silhouette", "davies_bouldin", "calinski_harabasz", "inertia", "cluster_balance"]
summary_rows = []

for m in metrics_order:
    s = df_k2[m]
    summary_rows.append({
        "metric": m,
        "n_total": len(df_k2),
        "n_available": s.notna().sum(),
        "mean": s.mean(),
        "median": s.median(),
        "minimum": s.min(),
        "maximum": s.max()
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

# Ghi ra CSV
summary_df.to_csv(f'{out_dir}/quality_summary.csv', index=False)


## 3. BẢNG 2: Chi tiết theo từng tháng

In [ ]:
cols = ["snapshot_date", "silhouette", "davies_bouldin", "calinski_harabasz", "inertia", "cluster_balance"]
detail_df = df_k2[cols].copy()
detail_df.reset_index(drop=True, inplace=True)
display(detail_df)


## 4. Trực quan hóa dữ liệu (5 Biểu đồ)

In [ ]:
snapshots = df_k2["snapshot_date"].dt.strftime('%Y-%m').tolist()
x_idx = range(len(snapshots))

def plot_line_metric(y_vals, metric_name, title, ylabel, plot_num, threshold=None):
    plt.figure(figsize=(10, 3.8))
    plt.plot(x_idx, y_vals, marker="o", color="#1f77b4", linewidth=2, label=f"{metric_name}")
    
    med_val = float(np.median(y_vals))
    plt.axhline(med_val, color="red", linestyle="--", linewidth=1.5, label=f"Median: {med_val:.4f}")
    
    if threshold is not None:
        plt.axhline(threshold, color="darkred", linestyle=":", linewidth=2, label=f"Threshold: {threshold}")
        
    plt.title(title, pad=15)
    plt.ylabel(ylabel)
    plt.xticks(x_idx, snapshots, rotation=45)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_line_metric(df_k2["silhouette"], "Silhouette", "Biểu đồ 7.1: Silhouette Score (PCA+KMeans) theo thời gian", "Score", 1)
plot_line_metric(df_k2["davies_bouldin"], "Davies-Bouldin", "Biểu đồ 7.2: Davies-Bouldin Score (PCA+KMeans) theo thời gian", "Score", 2)
plot_line_metric(df_k2["calinski_harabasz"], "Calinski-Harabasz", "Biểu đồ 7.3: Calinski-Harabasz Score (PCA+KMeans) theo thời gian", "Score", 3)
plot_line_metric(df_k2["inertia"], "Inertia", "Biểu đồ 7.4: Inertia (PCA+KMeans) theo thời gian", "Inertia", 4)
plot_line_metric(df_k2["cluster_balance"], "Cluster Balance", "Biểu đồ 7.5: Cluster Balance (PCA+KMeans) theo thời gian", "Ratio (Min/Max)", 5, threshold=0.05)


## 5. Khung kết luận 3 phần bắt buộc (Diagnostic Rubrics)

### Phần A (Phán quyết Kỹ thuật):
Mô hình PCA + K-Means (K=2) đạt điểm phân tách tốt nhờ không gian chiều được tinh gọn. **Silhouette Median** và **Davies-Bouldin** duy trì ở ngưỡng lý tưởng. Tuy nhiên, mức độ cân bằng cụm vẫn rất thấp, **Cluster Balance Median** đôi lúc trượt xuống dưới 0.05 (ngưỡng cảnh báo). Hiện tượng này giống với K-Means Baseline, chứng tỏ Asymmetric Outlier Isolation là đặc tính nội tại của dữ liệu.

### Phần B (Bản chất thị trường):
Nguyên nhân gốc rễ là do đặc tính phân phối đuôi dày (heavy tails) của thị trường chứng khoán Việt Nam (có những nhóm cổ phiếu cực đoan về thanh khoản và động lượng). Dù đã giảm qua 4 PCA components, những dị biệt tuyến tính này vẫn được bảo toàn. PCA không làm bóp méo hay che giấu sự bất đối xứng của thị trường, mà giúp thuật toán K-Means bắt tụ điểm nhiễu nhanh hơn.

### Phần C (Handoff cho Nhiệm vụ 10):
Với chất lượng Silhouette xuất sắc, mô hình PCA + K-Means cung cấp đường biên phân loại cực kỳ rõ nét trong không gian 4 chiều. Hiện tượng mất cân bằng chỉ là phản ánh cấu trúc thanh khoản lệch của TTCKVN, không phải lỗi mô hình. Phương án này hoàn toàn **đủ điều kiện bước vào vòng đánh giá cuối cùng** ở Nhiệm vụ 10 để so sánh đối đầu với K-Means Baseline và Ward.
